# Lab 1 — Ratings Data: Preprocessing + EDA
**ACM Workshop · Recommender Systems · Day 1 · 09:45–10:45 (60 min)**

**Goal of this lab:** understand the dataset the recommender will be built from.
Not "learn Pandas" — answer three questions:

1. **Who are my users?**  2. **What are my items?**  3. **What does the rating data look like?**

**You will produce:** dataset statistics · data-quality report · rating distribution · user activity · item popularity · cold-start counts · user–item matrix · sparsity · a clean train/test split saved to `data/processed/` (reused in Labs 2 and 3).

> Pipeline position: **Raw data → `EDA` → similarity → CF → MF → prediction → top-N → evaluation**

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20, "display.width", 140)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# --- locate (or download) MovieLens "latest-small" -----------------------------------------
URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
def find_data():
    for base in ["data", "../data", "../../data", "../../../data"]:
        p = Path(base) / "ml-latest-small"
        if (p / "ratings.csv").exists():
            return p
    Path("data").mkdir(exist_ok=True)
    print("Downloading MovieLens latest-small (~1 MB)...")
    urllib.request.urlretrieve(URL, "data/ml.zip")
    zipfile.ZipFile("data/ml.zip").extractall("data")
    return Path("data/ml-latest-small")

DATA = find_data()
PROC = DATA.parent / "processed"; PROC.mkdir(exist_ok=True)
FIG  = DATA.parent.parent / "figures"; FIG.mkdir(exist_ok=True)
print("data folder :", DATA.resolve())

## 1. Load the data
MovieLens *latest-small* (GroupLens, University of Minnesota): ~100 k ratings from 610 users on ~9.7 k movies, collected 1996–2018.
Each user has rated **at least 20 movies** — GroupLens already filtered the data this way.

In [ ]:
ratings = pd.read_csv(DATA / "ratings.csv")
movies  = pd.read_csv(DATA / "movies.csv")
ratings.head()

In [ ]:
print("ratings:", ratings.shape, "| movies:", movies.shape)
ratings.info()

In [ ]:
movies.head()

In [ ]:
n_users, n_items, n_ratings = ratings.userId.nunique(), ratings.movieId.nunique(), len(ratings)
print(f"users   : {n_users:,}")
print(f"movies  : {n_items:,}  (catalogue has {len(movies):,}; some are never rated)")
print(f"ratings : {n_ratings:,}")

**Concept check.** What do the columns mean?
`userId` = who · `movieId` = what · `rating` = how much (explicit feedback, 0.5–5.0 in half-star steps) · `timestamp` = when (Unix seconds).
Titles and genres live in a *separate* table → we will need a `merge`.

## 2. Data quality

In [ ]:
print("missing values per column:\n", ratings.isnull().sum(), "\n")
print("fully duplicated rows        :", ratings.duplicated().sum())
print("duplicate (user, movie) pairs:", ratings.duplicated(["userId", "movieId"]).sum())
ratings.describe().round(2)

**Questions** — answer in a markdown cell:
- Missing values? Duplicates? → *None, this dataset is already clean.*
- Min / max / mean rating?
- Why do we check **(user, movie) duplicates** specifically? *Because the matrix cell `R[u, i]` must hold exactly one value.*
- Do all ratings in `movies.csv` have a matching movie? Check below.

In [ ]:
orphan = set(ratings.movieId) - set(movies.movieId)
unrated = set(movies.movieId) - set(ratings.movieId)
print("rated movies missing from movies.csv:", len(orphan))
print("catalogue movies nobody has rated   :", len(unrated))

## 3. Rating distribution
Before modelling, look at what a rating *is* in this data.

In [ ]:
counts = ratings.rating.value_counts().sort_index()
print(counts)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].bar(counts.index.astype(str), counts.values, color="#2b6cb0")
ax[0].set(title="Rating frequency", xlabel="rating", ylabel="# ratings")
ax[1].bar(["mean", "median"], [ratings.rating.mean(), ratings.rating.median()], color="#dd6b20")
ax[1].set(title=f"Mean = {ratings.rating.mean():.2f}", ylim=(0, 5))
plt.tight_layout(); plt.savefig(FIG / "lab1_rating_dist.png"); plt.show()

**Observations to discuss**
- Ratings are **skewed positive**: 4.0 is the most common value, and whole stars are more common than half stars.
- People mostly rate things they *chose to watch* → **selection bias**. A rating of 3.5 is therefore not "average movie", it is "average among movies someone picked".
- The global mean (~3.5) is already a decent predictor. **Any model must beat it** (we will use it as a baseline in Labs 2–3).

## 4. User analysis — who are my users?

In [ ]:
per_user = ratings.groupby("userId").size().rename("n_ratings")
print(per_user.describe().round(1))
print("\nTop-5 most active users:\n", per_user.nlargest(5))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].hist(per_user, bins=60, color="#2b6cb0"); ax[0].set(title="Ratings per user", xlabel="# ratings", ylabel="# users")
ax[1].hist(np.log10(per_user), bins=40, color="#2b6cb0"); ax[1].set(title="Same, log10 x-axis", xlabel="log10(# ratings)")
plt.tight_layout(); plt.savefig(FIG / "lab1_user_activity.png"); plt.show()

share = per_user.sort_values(ascending=False).cumsum() / per_user.sum()
print(f"Top 10% of users ({int(.1*len(per_user))}) produce {share.iloc[int(.1*len(per_user))-1]:.0%} of all ratings")

In [ ]:
# Do heavy raters rate differently? Mean rating per user.
user_mean = ratings.groupby("userId").rating.mean()
fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.hist(user_mean, bins=30, color="#38a169"); ax.set(title="Average rating given by each user", xlabel="user mean rating", ylabel="# users")
plt.tight_layout(); plt.savefig(FIG / "lab1_user_mean.png"); plt.show()
print(f"user means range from {user_mean.min():.2f} to {user_mean.max():.2f}")

**Take-aways**
- The distribution is **heavy-tailed**: a few "power users" rate thousands of movies, the typical user rates ~70 (median).
- Users have different **rating habits** — some are generous (mean 4.5), some harsh (mean 2.5). This is called **user bias**, and is why Lab 2 *mean-centres* ratings and why SVD adds a bias term $b_u$.

## 5. Item analysis — what are my items?

In [ ]:
per_item = ratings.groupby("movieId").size().rename("n_ratings")
item_stats = (ratings.groupby("movieId").rating.agg(["mean", "count"])
              .join(movies.set_index("movieId")[["title", "genres"]]))
print(per_item.describe().round(1))
print("\nMost-rated movies:")
item_stats.nlargest(8, "count")[["title", "count", "mean"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
s = per_item.sort_values(ascending=False).values
ax[0].plot(s, color="#c53030"); ax[0].set(title="Long tail: ratings per movie (ranked)", xlabel="movie rank", ylabel="# ratings")
ax[1].hist(per_item.clip(upper=50), bins=50, color="#c53030"); ax[1].set(title="Ratings per movie (clipped at 50)", xlabel="# ratings")
plt.tight_layout(); plt.savefig(FIG / "lab1_long_tail.png"); plt.show()

In [ ]:
# Careful: "highest rated" without a minimum count is misleading.
print("Top-rated, NO minimum count:")
display(item_stats.nlargest(5, "mean")[["title", "count", "mean"]])
print("Top-rated, at least 50 ratings:")
item_stats[item_stats["count"] >= 50].nlargest(5, "mean")[["title", "count", "mean"]]

**Take-aways**
- The **long tail**: a handful of blockbusters get hundreds of ratings; most movies get very few.
- Movies with 1–2 ratings have **unreliable averages** (a single 5★ = "best movie ever"). Every recommender needs a *support threshold* or *shrinkage*.
- This is the root of **popularity bias**: popular movies have more data, so models know them better and recommend them more.

## 6. Cold start — quantify it
> *What happens when a new user joins, or a new movie has no ratings?*

In [ ]:
for k in [1, 5, 10, 20]:
    print(f"movies with <= {k:>2} ratings: {(per_item <= k).sum():>5,}  ({(per_item <= k).mean():.1%} of rated movies)")
print(f"\nmovies in the catalogue with ZERO ratings: {len(movies) - len(per_item)}")
print(f"users with < 20 ratings: {(per_user < 20).sum()}  (dataset guarantees >= 20 — real systems do not!)")

**Discuss (2 min, pairs):** A collaborative-filtering model can only recommend things that *someone has rated*. What should Netflix do for (a) a brand-new user, (b) a brand-new film? *(Hints: popularity, content features, asking onboarding questions → Day 2 / hybrids.)*

## 7. Genres & time (bonus context)

In [ ]:
movies["year"] = movies.title.str.extract(r"\((\d{4})\)\s*$").astype(float)
movies["genre_list"] = movies.genres.str.split("|")
g = (ratings.merge(movies[["movieId", "genre_list"]], on="movieId")
            .explode("genre_list").groupby("genre_list").rating.agg(["count", "mean"]).sort_values("count", ascending=False))
g.round(2).head(12)

In [ ]:
ratings["dt"] = pd.to_datetime(ratings.timestamp, unit="s")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ratings.dt.dt.year.value_counts().sort_index().plot(kind="bar", ax=ax[0], color="#805ad5"); ax[0].set(title="Ratings per year", xlabel="")
g["count"].head(12)[::-1].plot(kind="barh", ax=ax[1], color="#805ad5"); ax[1].set(title="Ratings per genre", ylabel="")
plt.tight_layout(); plt.savefig(FIG / "lab1_genre_time.png"); plt.show()
print("date range:", ratings.dt.min().date(), "→", ratings.dt.max().date())

## 8. The user–item matrix and sparsity  ⭐ *the key concept of the lab*
Rows = users, columns = movies, values = ratings, **NaN = unknown**.

In [ ]:
user_item = ratings.pivot_table(index="userId", columns="movieId", values="rating")
print("matrix shape:", user_item.shape)
cells = user_item.shape[0] * user_item.shape[1]
print(f"possible cells : {cells:,}")
print(f"observed cells : {user_item.notna().sum().sum():,}")
print(f"sparsity       : {1 - user_item.notna().sum().sum()/cells:.2%} of the matrix is EMPTY")
user_item.iloc[:8, :8]

In [ ]:
# Visualise: densest corner of the matrix vs a random corner
top_u = per_user.nlargest(120).index; top_i = per_item.nlargest(120).index
rng = np.random.default_rng(0)
rnd_u = rng.choice(user_item.index, 120, replace=False); rnd_i = rng.choice(user_item.columns, 120, replace=False)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
for a, (u, i, t) in zip(ax, [(top_u, top_i, "120 most active users × 120 most popular movies"),
                              (rnd_u, rnd_i, "120 random users × 120 random movies")]):
    sub = user_item.loc[u, i].notna().values
    a.imshow(sub, cmap="Blues", aspect="auto", interpolation="nearest")
    a.set(title=f"{t}\n{sub.mean():.1%} filled", xticks=[], yticks=[])
plt.tight_layout(); plt.savefig(FIG / "lab1_sparsity.png"); plt.show()

**Why is it so empty?** Nobody can watch 9,000 movies. Even the most engaged users have seen a tiny fraction of the catalogue.
A recommender's job is exactly this: **predict the empty cells** — and then rank them.

> *This is the conceptual bridge to the next session: how can we predict a missing cell using other users' behaviour?*

## 9. Pre-processing decisions & the train/test split
Decisions we make (and why):

| Decision | Choice | Reason |
|---|---|---|
| Duplicates / NaNs | none to remove | checked in §2 |
| Drop rare movies? | **No** for the split; Lab 2 applies a *support threshold at recommendation time* | dropping from the data changes the evaluation problem |
| Rating scale | keep 0.5–5.0 | Surprise `Reader(rating_scale=(0.5, 5))` |
| Split | random 80 / 20 with `random_state=42`, **same split in all labs** | comparable RMSE numbers |

> ⚠️ Random splits are the simplest option. A *temporal* split (train on past, test on future) is more realistic but gives different numbers — an extension for the curious.

In [ ]:
from sklearn.model_selection import train_test_split
train, test = train_test_split(ratings[["userId", "movieId", "rating"]], test_size=0.2, random_state=42)
train.to_csv(PROC / "train.csv", index=False); test.to_csv(PROC / "test.csv", index=False)
ratings[["userId", "movieId", "rating"]].to_csv(PROC / "ratings_clean.csv", index=False)
movies[["movieId", "title", "genres", "year"]].to_csv(PROC / "movies_clean.csv", index=False)

print(f"train: {len(train):,}   test: {len(test):,}")
# How many test ratings involve a user / movie never seen in training? (a mini cold-start measurement)
print("test rows whose user  is unseen in train:", (~test.userId.isin(train.userId)).sum())
print("test rows whose movie is unseen in train:", (~test.movieId.isin(train.movieId)).sum(),
      f"({(~test.movieId.isin(train.movieId)).mean():.1%})")

## 10. Exercises (finish in the last 10 min)

**E1.** Which genre has the highest *mean* rating among genres with ≥ 1,000 ratings? Which has the lowest?
**E2.** Find the 5 users whose mean rating is lowest (min. 50 ratings). How would that affect a naive "average of neighbours" prediction?
**E3.** What fraction of all ratings goes to the 100 most-rated movies?

In [ ]:
# E1
# TODO: write your code here


In [ ]:
# E2
# TODO: write your code here


In [ ]:
# E3
# TODO: write your code here


## ✅ Lab 1 deliverable — checklist
- [x] dataset statistics · [x] missing-value / duplicate check · [x] rating distribution
- [x] user activity · [x] item popularity (long tail) · [x] cold-start counts
- [x] user–item matrix + sparsity · [x] saved `data/processed/{train,test}.csv`

**Next (11:00): Collaborative Filtering theory** — *"How can we fill those empty cells using other users' behaviour?"*

# Lab 2 — Build a Working Collaborative-Filtering Recommender
**ACM Workshop · Recommender Systems · Day 1 · 11:45–12:45 (60 min)**

Just taught: *similar users have similar tastes* →
$$\hat r_{ui}=\bar r_u+\frac{\sum_{v\in N(u)} \mathrm{sim}(u,v)\,(r_{vi}-\bar r_v)}{\sum_{v\in N(u)}|\mathrm{sim}(u,v)|}$$

**You will build** (with NumPy/scikit-learn only — no recommender library yet):
1. similarity from a toy example → then from real data
2. **user-based CF**: neighbours → predictions → Top-N for user 10
3. **item-based CF**: similar movies → predictions
4. an honest **evaluation** (RMSE vs baselines) and an experiment on **k**
5. a **cold-start** demo — pretend to be a brand-new user

> Pipeline position: Raw data → EDA → **similarity → collaborative filtering** → MF → prediction → top-N → evaluation

## 0. Setup & data (the split from Lab 1)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_columns", 20, "display.width", 140, "display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def find_proc():
    for base in ["data", "../data", "../../data", "../../../data"]:
        if (Path(base) / "processed" / "train.csv").exists(): return Path(base) / "processed"
    raise FileNotFoundError("Run Lab 1 first (it creates data/processed/train.csv).")
PROC = find_proc(); FIG = PROC.parent.parent / "figures"; FIG.mkdir(exist_ok=True)

train  = pd.read_csv(PROC / "train.csv")
test   = pd.read_csv(PROC / "test.csv")
movies = pd.read_csv(PROC / "movies_clean.csv").set_index("movieId")
title  = movies.title
print(len(train), "train ratings |", len(test), "test ratings")

## 1. Similarity on a toy example (the slide example)
Three users, three movies. Who is similar to A?

In [ ]:
toy = pd.DataFrame({"LOTR": [5, 5, 1], "Matrix": [5, 4, 1], "Titanic": [1, 1, 5]}, index=list("ABC"))
print(toy, "\n")
sim = pd.DataFrame(cosine_similarity(toy), index=toy.index, columns=toy.index).round(3)
print("cosine similarity:\n", sim)

By hand for A and B: $\cos = \dfrac{5\cdot5+5\cdot4+1\cdot1}{\sqrt{51}\,\sqrt{42}} = \dfrac{46}{46.28}\approx 0.994$.
**But** look at A vs C: cosine is still ~0.40 although they disagree on everything. Raw cosine on ratings is high whenever all numbers are positive.
The fix: **mean-centre** each user's ratings first (subtract the user's average) — then "above average" is positive and "below average" is negative.

In [ ]:
centred = toy.sub(toy.mean(axis=1), axis=0)
print(centred.round(2), "\n")
print("cosine on centred ratings:\n", pd.DataFrame(cosine_similarity(centred), index=toy.index, columns=toy.index).round(3))

→ Now A and B are almost identical (+0.97) and A and C are opposites (−1.0). This is (nearly) the **Pearson correlation**.

## 2. User–item matrix from the training data
Same as Lab 1 but in plain NumPy for speed. `NaN` = unknown. **Only training ratings** go in — the test ratings stay hidden.

In [ ]:
all_items = np.sort(pd.concat([train.movieId, test.movieId]).unique())
users     = np.sort(train.userId.unique())
u_idx = {u: k for k, u in enumerate(users)}; i_idx = {i: k for k, i in enumerate(all_items)}

R = np.full((len(users), len(all_items)), np.nan)
R[train.userId.map(u_idx), train.movieId.map(i_idx)] = train.rating.values

M    = ~np.isnan(R)                      # mask: True where a rating is known
mu_u = np.nanmean(R, axis=1)             # each user's mean rating
Rc   = np.where(M, R - mu_u[:, None], 0.0)   # mean-centred, unknown -> 0
n_item = M.sum(axis=0)                   # popularity (training ratings per movie)
print("R:", R.shape, "| known cells:", M.sum(), f"| sparsity {1 - M.mean():.2%}")

## 3. User-based CF
### 3.1 Similarity between users
Cosine similarity of the **mean-centred** rows. Unknown cells are 0, i.e. "no opinion", so they do not contribute to the dot product.

In [ ]:
norm_u = np.linalg.norm(Rc, axis=1)
Mf = M.astype(float)
overlap_uu = Mf @ Mf.T                                     # number of movies each pair of users has in common

def build_S_uu(shrink=None):
    S = (Rc @ Rc.T) / (np.outer(norm_u, norm_u) + 1e-9)    # cosine of mean-centred rows
    if shrink: S = S * np.minimum(1, overlap_uu / shrink)   # significance weighting (optional)
    np.fill_diagonal(S, 0)                                  # a user is not their own neighbour
    return S
S_uu = build_S_uu()

def user_sims(vec):
    # similarity of ANY (possibly partial / brand-new) rating vector to every user in the matrix
    m = ~np.isnan(vec); v = np.where(m, vec - np.nanmean(vec), 0.0)
    return Rc @ v / (norm_u * np.linalg.norm(v) + 1e-9)

TARGET = 10
s = S_uu[u_idx[TARGET]]
nn = np.argsort(-s)[:10]
pd.DataFrame({"neighbour userId": users[nn], "similarity": s[nn].round(3),
              "# movies in common": overlap_uu[u_idx[TARGET], nn].astype(int), "# ratings": M[nn].sum(1)})

**Look carefully**: some neighbours share only a handful of movies with user 10 → the similarity is *unreliable*.
A common fix is **significance weighting** — shrink similarity when the overlap is small: $\mathrm{sim}' = \mathrm{sim}\cdot\min(1, n_{uv}/50)$. We will try it in an exercise.

### 3.2 What does user 10 like? (so we can judge the recommendations)

In [ ]:
def history(u, n=8):
    row = R[u_idx[u]]; ids = all_items[~np.isnan(row)]
    h = pd.DataFrame({"title": title.reindex(ids).values, "rating": row[~np.isnan(row)]}, index=ids)
    return h.sort_values("rating", ascending=False).head(n)
print(f"User {TARGET} has {M[u_idx[TARGET]].sum()} training ratings, mean {mu_u[u_idx[TARGET]]:.2f}. Favourites:")
history(TARGET)

### 3.3 Predict + recommend
$$\hat r_{ui}=\bar r_u+\frac{\sum_{v\in N_k(u,i)} s_{uv}\,(r_{vi}-\bar r_v)}{\sum_{v\in N_k(u,i)} |s_{uv}|}$$
$N_k(u,i)$ = the **k most similar users *who rated movie i*** (only positively-similar ones are used).

In [ ]:
raters = [np.where(M[:, j])[0] for j in range(M.shape[1])]       # who rated each movie (training data)

def predict_ui(s, j, k=20):
    # s: similarity of the target user to all users; j: movie column.  Returns (prediction offset, support)
    r = raters[j]
    if len(r) == 0: return np.nan, 0
    sr = s[r]
    if len(r) > k:
        top = np.argpartition(-sr, k - 1)[:k]; r, sr = r[top], sr[top]
    pos = sr > 0
    if not pos.any(): return np.nan, 0
    r, sr = r[pos], sr[pos]
    return (sr @ Rc[r, j]) / np.abs(sr).sum(), int(pos.sum())     # weighted average of centred ratings

def recommend_from_sims(s, base, seen, k=20, n=5, min_pop=20, min_support=3):
    # s: similarities to all users, base: the user's mean rating, seen: mask of movies already rated
    out = []
    for j in np.where(~seen & (n_item >= min_pop))[0]:
        off, sup = predict_ui(s, j, k)
        if sup >= min_support: out.append((j, base + off, sup))
    out = sorted(out, key=lambda t: -t[1])[:n]
    return pd.DataFrame({"title": [title[all_items[j]] for j, _, _ in out], "pred_rating": [round(min(5, p), 2) for _, p, _ in out],
                         "# neighbours used": [c for _, _, c in out], "# ratings overall": [n_item[j] for j, _, _ in out]},
                        index=[all_items[j] for j, _, _ in out])

def recommend_user_cf(u, **kw):
    i = u_idx[u]; return recommend_from_sims(S_uu[i], mu_u[i], M[i], **kw)

print(f"Top-5 for user {TARGET}  (k=20 neighbours)")
recommend_user_cf(TARGET, k=20)

**What are `min_pop` and `min_support` doing?** Without them, the highest predictions are movies that *one* neighbour rated 5★. They are a guard against the unreliable-average problem from Lab 1 (§5).
Try `min_pop=0, min_support=1` to see the effect → it is dramatic.

In [ ]:
print("WITHOUT the support guards (min_pop=0, min_support=1):")
recommend_user_cf(TARGET, k=20, min_pop=0, min_support=1)

### 3.4 Discussion experiment: how many neighbours?

In [ ]:
lists = {k: recommend_user_cf(TARGET, k=k, n=5, min_support=1 if k == 2 else 3) for k in (2, 20, 50)}
for k, df in lists.items():
    print(f"\n--- k = {k} ---"); print(df[["title", "pred_rating", "# neighbours used"]].to_string())

**Discuss in pairs (3 min):**
- With **k = 2** the list depends on just two people → noisy; the predictions are inflated (4.7+) because two strangers' opinions are not averaged with anything.
- **k = 20 and k = 50 give the *identical* list. Why?** Look at the last column: each candidate was rated by only 3–10 of the positively-similar users. When fewer than *k* neighbours have rated a movie, increasing *k* changes nothing — **sparsity, not *k*, limits the neighbourhood.**
- So *k* trades **variance** (small k: noisy) against **bias** (large k: less personal) — but only until the data runs out. We measure the effect on error in §5.

## 4. Item-based CF
> "Which movies are similar to the movies I already liked?"

Item–item similarity: cosine of the **columns** of the (user-mean-centred) matrix — the *adjusted cosine*.
We restrict to movies with ≥ 10 training ratings (similarity from 1–2 ratings is meaningless, and the full matrix would not fit in memory).

In [ ]:
MIN_ITEM = 10
keep = np.where(n_item >= MIN_ITEM)[0]
print(f"{len(keep)} movies have >= {MIN_ITEM} training ratings (of {len(all_items)})")
Xi = Rc[:, keep]
nrm = np.linalg.norm(Xi, axis=0) + 1e-9
S_item = (Xi.T @ Xi) / np.outer(nrm, nrm)
np.fill_diagonal(S_item, 0)
kept_ids = all_items[keep]; k_idx = {i: j for j, i in enumerate(kept_ids)}

def similar_movies(movie_id, n=8):
    s = pd.Series(S_item[k_idx[movie_id]], index=kept_ids).nlargest(n)
    return pd.DataFrame({"title": title.reindex(s.index).values, "similarity": s.round(3).values,
                         "# ratings": n_item[[i_idx[i] for i in s.index]]}, index=s.index)

print("Movies most similar to:", title[1]); display(similar_movies(1))
print("Movies most similar to:", title[2571]); display(similar_movies(2571))

No genre information was used — *Toy Story's* neighbours are other animated/family films purely because **the same people rated them similarly**. That is the "collaborative" part.

In [ ]:
# for every user: which (kept) movies they rated and their centred ratings
rated_k = [np.where(M[u, keep])[0] for u in range(len(users))]
rated_c = [Rc[u, keep][rated_k[u]] for u in range(len(users))]

def predict_item_based(u, j, k=20):
    # j: index in the `keep` space. Neighbours = the k movies MOST SIMILAR TO j among those user u has rated.
    idx = rated_k[u]
    if len(idx) == 0: return np.nan, 0
    sj = S_item[j, idx]; c = rated_c[u]
    if len(idx) > k:
        top = np.argpartition(-sj, k - 1)[:k]; sj, c = sj[top], c[top]
    pos = sj > 0
    if not pos.any(): return np.nan, 0
    return (sj[pos] @ c[pos]) / sj[pos].sum(), int(pos.sum())

def recommend_item_cf(u, k=20, n=5, min_support=3, min_pop=30):
    ui = u_idx[u]; out = []
    for j in np.where(~M[ui, keep] & (n_item[keep] >= min_pop))[0]:
        off, sup = predict_item_based(ui, j, k)
        if sup >= min_support: out.append((j, mu_u[ui] + off, sup))
    out = sorted(out, key=lambda t: -t[1])[:n]
    ids = [kept_ids[j] for j, _, _ in out]
    return pd.DataFrame({"title": title.reindex(ids).values, "pred_rating": [round(min(5, p), 2) for _, p, _ in out],
                         "# similar movies used": [c for _, _, c in out]}, index=ids)
print(f"Item-based Top-5 for user {TARGET}"); recommend_item_cf(TARGET)

Compare with §3.3. Item-based lists tend to be *"more of what you already like"* (safer, more stable), user-based lists can be more surprising.

## 5. Evaluation — do the numbers agree with our intuition?
Predict every **test** rating, compare with the truth. Always include **baselines**: if you cannot beat "predict the average", you have not built a recommender.

In [ ]:
test_u = test.userId.map(u_idx).values; test_i = test.movieId.map(i_idx).values; y = test.rating.values
def evaluate(pred, name):
    p = np.where(np.isnan(pred), mu_u[test_u], pred).clip(0.5, 5)      # fall back to the user's mean when CF has no answer
    return {"model": name, "RMSE": np.sqrt(np.mean((y - p) ** 2)), "MAE": np.mean(np.abs(y - p)), "coverage": 1 - np.isnan(pred).mean()}

def pred_user_cf(k, S=None):
    S = S_uu if S is None else S
    return np.array([mu_u[u] + predict_ui(S[u], j, k)[0] for u, j in zip(test_u, test_i)])

kept_pos = {j: p for p, j in enumerate(keep)}
def pred_item_cf(k):
    out = np.full(len(test_u), np.nan)
    for n_, (u, j) in enumerate(zip(test_u, test_i)):
        if j in kept_pos: out[n_] = mu_u[u] + predict_item_based(u, kept_pos[j], k)[0]
    return out

g_mean = train.rating.mean()
item_mean = np.where(n_item > 0, np.nansum(R, 0) / np.maximum(n_item, 1), g_mean)
rows = [evaluate(np.full(len(y), g_mean), "baseline: global mean"),
        evaluate(mu_u[test_u], "baseline: user mean"),
        evaluate(item_mean[test_i], "baseline: item mean"),
        evaluate(pred_user_cf(20), "user-based CF (k=20)"),
        evaluate(pred_user_cf(50), "user-based CF (k=50)"),
        evaluate(pred_item_cf(20), "item-based CF (k=20)")]
res = pd.DataFrame(rows).set_index("model"); res.round(4)

**Read it honestly.** Similarity-based CF beats every baseline, but the gap to the *user-mean* baseline (~0.06 RMSE) is smaller than people expect. *Coverage* differs too: item-based CF only answers for movies with ≥ 10 ratings (≈75 % of test pairs; the rest fall back to the user mean), so compare RMSEs with care.
Part of the signal is just "this user is generous / this movie is good" (**biases**) — matrix factorization (next session) models those explicitly.

In [ ]:
ks = [1, 2, 5, 10, 20, 50, 100, 200]
curve = pd.DataFrame([evaluate(pred_user_cf(k), f"k={k}") | {"k": k} for k in ks]).set_index("k")
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(curve.index, curve.RMSE, "o-", color="#2b6cb0"); ax.set_xscale("log")
ax.axhline(res.loc["baseline: user mean", "RMSE"], ls="--", color="gray", label="user-mean baseline"); ax.legend()
ax.set(xlabel="k (neighbours)", ylabel="test RMSE", title="User-based CF: effect of k")
plt.tight_layout(); plt.savefig(FIG / "lab2_k_curve.png"); plt.show()
curve[["RMSE", "MAE", "coverage"]].round(4)

**Interpretation.** With *k = 1* we trust a single stranger → worse than predicting the global mean. Error drops fast up to *k ≈ 10–20*, then **flattens** (diminishing returns: extra neighbours have tiny similarity weights).
Choosing *k* is **hyper-parameter tuning**: do it on a *validation* set, never on the test set (here we show the test curve for teaching only).

## 6. Cold start — pretend to be a new user
A brand-new user has *no* row in the matrix. We ask for 3–5 favourite movies and run CF on that partial vector.

In [ ]:
def find(q): return movies[movies.title.str.contains(q, regex=False)].index[0]
my_ratings = {find("Matrix, The (1999)"): 5, find("Star Wars: Episode IV"): 5, find("Alien (1979)"): 4.5, find("Toy Story (1995)"): 2}
vec = np.full(R.shape[1], np.nan)
for mid, r_ in my_ratings.items(): vec[i_idx[mid]] = r_
print("I am a new user. My ratings:", {title[m]: r for m, r in my_ratings.items()})

new_s = user_sims(vec)
print("\nRecommendations for the new user:")
recommend_from_sims(new_s, np.nanmean(vec), ~np.isnan(vec), k=30, n=8, min_pop=50)

With only four ratings we already get a (rough) personalised list. With **zero** ratings CF has nothing to work with — the standard fallback is **popularity**:

In [ ]:
# Popularity fallback: Bayesian-damped mean ("IMDb-style" weighted rating)
C, m_ = g_mean, 30
agg = train.groupby("movieId").rating.agg(["mean", "count"])
agg["score"] = (agg["count"] * agg["mean"] + m_ * C) / (agg["count"] + m_)
agg.nlargest(8, "score").assign(title=lambda d: title.reindex(d.index).values)[["title", "count", "mean", "score"]].round(2)

## 7. Exercises (pick one, 10 min)
**E1 — Significance weighting.** `build_S_uu(shrink)` multiplies each similarity by `min(1, overlap/shrink)`. Predict first: will RMSE at k=20 improve? Then run it. *(Ideas from papers must be validated on your data!)*
**E2 — Explain a recommendation.** For user 10's #1 recommendation, list which neighbours contributed and what they rated.
**E3 — Another user.** Run the Top-5 for a user with *few* ratings and one with *thousands* (Lab 1: users 414, 599). What differs?

In [ ]:
# E2 — explainability: who drove the #1 recommendation?
# TODO: write your code here


In [ ]:
# E1 — significance weighting: shrink similarities that rest on few co-rated movies
# TODO: write your code here


## 8. Limitations we just *felt* (→ next theory session)
| Observed | Name | What fixes it |
|---|---|---|
| Neighbours share few movies | **sparsity** | compress users/items into dense vectors |
| §3.3 junk without support filters | **unreliable similarity / popularity bias** | regularisation |
| New user needed hand-built vector | **cold start** | content features, hybrids |
| `user_cf_matrix` loops over every user; item-sim matrix is $n^2$ | **scalability** | learn a *small* number of factors |
| Predictions only as good as "bias + neighbours" | **bias is not modelled** | $b_u$, $b_i$ in matrix factorization |

## ✅ Lab 2 deliverable — checklist
- [x] user–user and item–item similarity · [x] Top-5 for user 10 (user-based and item-based) · [x] k experiment · [x] RMSE vs baselines · [x] new-user demo

**Next (13:30): Matrix Factorization — SVD / ALS.** *"What if we discover hidden factors that explain preferences instead of comparing users one by one?"*

# Lab 3 — Matrix Factorization Recommender with Surprise (SVD)
**ACM Workshop · Recommender Systems · Day 1 · 14:15–15:45 (90 min)**

Just taught: $R\approx PQ^\top$, i.e. $\hat r_{ui}=\mu+b_u+b_i+p_u^\top q_i$, learned by minimising regularised squared error.

**You will build** the main technical deliverable of Day 1:
1. load the *same* train/test split into **Surprise** · 2. train **SVD** · 3. predict & read the predictions
4. evaluate **RMSE / MAE** against baselines and the Lab-2 models · 5. **open the model**: $P$, $Q$, biases
6. **Top-N** recommendations for user 10 · 7. a **hyper-parameter experiment** (`n_factors`, multi-seed) + a small tuning grid
8. **rating prediction ≠ ranking**: Precision@10 / Recall@10 vs a popularity baseline
9. *(optional)* **ALS** from scratch in 25 lines

> Pipeline: Raw → EDA → similarity → CF → **matrix factorization → prediction → top-N → evaluation**

## 0. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from surprise import Dataset, Reader, SVD, BaselineOnly, KNNWithMeans, NormalPredictor, accuracy
from surprise.model_selection import GridSearchCV

pd.set_option("display.max_columns", 20, "display.width", 140, "display.max_colwidth", 60)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def find_proc():
    for base in ["data", "../data", "../../data", "../../../data"]:
        if (Path(base) / "processed" / "train.csv").exists(): return Path(base) / "processed"
    raise FileNotFoundError("Run Lab 1 first (it creates data/processed/train.csv).")
PROC = find_proc(); FIG = PROC.parent.parent / "figures"; FIG.mkdir(exist_ok=True)
train  = pd.read_csv(PROC / "train.csv"); test = pd.read_csv(PROC / "test.csv")
movies = pd.read_csv(PROC / "movies_clean.csv").set_index("movieId"); title = movies.title
import surprise; print("scikit-surprise", surprise.__version__, "| train", len(train), "| test", len(test))

## 1. Convert Pandas → Surprise
Surprise needs a `Reader` that knows the **rating scale**. MovieLens-small uses **0.5–5.0** (the 1–5 scale of MovieLens-100k would be wrong here!).

We reuse the Lab-1 split so that numbers are comparable with Lab 2: build the training set from `train.csv`, and the test set as a list of `(user, item, true_rating)` tuples.

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(train[["userId", "movieId", "rating"]], reader)
trainset = data.build_full_trainset()                       # Surprise's internal structure (maps raw ids -> inner ids)
testset  = list(test[["userId", "movieId", "rating"]].itertuples(index=False, name=None))

print("users:", trainset.n_users, "| items:", trainset.n_items, "| ratings:", trainset.n_ratings)
print("global mean μ:", round(trainset.global_mean, 4))
print("first test tuple (uid, iid, true rating):", testset[0])

## 2. Baselines — what must we beat?
- `NormalPredictor`: random ratings drawn from the training distribution (floor).
- `BaselineOnly`: $\hat r_{ui}=\mu+b_u+b_i$ — **no personalisation beyond biases**, fitted by ALS. Surprisingly strong!

In [ ]:
results = {}
def run(name, algo):
    t = time.time(); algo.fit(trainset); pred = algo.test(testset)
    results[name] = {"RMSE": accuracy.rmse(pred, verbose=False), "MAE": accuracy.mae(pred, verbose=False), "fit+predict (s)": round(time.time() - t, 1)}
    return algo, pred

run("random (NormalPredictor)", NormalPredictor())
run("baseline  μ + b_u + b_i", BaselineOnly(verbose=False))
pd.DataFrame(results).T.round(4)

## 3. Train SVD
```python
SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, biased=True, random_state=...)
```
| parameter | meaning |
|---|---|
| `n_factors` | dimension $k$ of $p_u,q_i$ — model capacity |
| `n_epochs` | passes of SGD over the training ratings |
| `lr_all` | learning rate (step size) |
| `reg_all` | $\lambda$, L2 regularisation strength |
| `biased` | include $\mu, b_u, b_i$ (True by default) |

In [ ]:
svd = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
svd, predictions = run("SVD (50 factors)", svd)      # fit on trainset, then predict testset
print("predictions is a list of", len(predictions), "Prediction objects. The first one:\n", predictions[0])

## 4. Look at the predictions

In [ ]:
pred_df = pd.DataFrame([(p.uid, p.iid, p.r_ui, p.est) for p in predictions], columns=["userId", "movieId", "actual", "predicted"])
pred_df["title"] = title.reindex(pred_df.movieId).values
pred_df["abs_error"] = (pred_df.actual - pred_df.predicted).abs()
print("A random sample of test predictions:")
display(pred_df.sample(8, random_state=1).round(2))
print("\nBiggest mistakes:")
pred_df.nlargest(5, "abs_error").round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].scatter(pred_df.actual + np.random.default_rng(0).uniform(-.12, .12, len(pred_df)), pred_df.predicted, s=2, alpha=.15, color="#2b6cb0")
ax[0].plot([0.5, 5], [0.5, 5], "r--"); ax[0].set(xlabel="actual rating", ylabel="predicted rating", title="Predicted vs actual (jittered)")
ax[1].hist(pred_df.actual - pred_df.predicted, bins=50, color="#38a169"); ax[1].set(title="Error distribution (actual − predicted)", xlabel="error")
plt.tight_layout(); plt.savefig(FIG / "lab3_pred_vs_actual.png"); plt.show()

**Read the plot:** predictions are *compressed towards the middle* (the model rarely says 0.5 or 5.0 — it is hedging, which minimises squared error). That is why RMSE ≈ 0.88 although ratings span 0.5–5.

## 5. Evaluate: RMSE and MAE
$$\mathrm{RMSE}=\sqrt{\tfrac1N\sum(r-\hat r)^2}\qquad \mathrm{MAE}=\tfrac1N\sum|r-\hat r|$$
RMSE punishes large errors more; MAE is the typical error in "stars".

In [ ]:
print("SVD  RMSE:", round(accuracy.rmse(predictions, verbose=False), 4), "| MAE:", round(accuracy.mae(predictions, verbose=False), 4))
# Compare with the Lab-2 style neighbourhood models, now via Surprise's built-in KNN
run("user-KNN (k=40, Pearson)", KNNWithMeans(k=40, sim_options={"name": "pearson", "user_based": True}, verbose=False))
run("item-KNN (k=40, cosine)",  KNNWithMeans(k=40, sim_options={"name": "cosine",  "user_based": False}, verbose=False))
tab = pd.DataFrame(results).T; tab.round(4)

**Honest reading of this table** — this is a *teaching point*, not a flaw:
- All personalised models land close to each other; the **bias baseline** already captures most of the signal.
- SVD gains over the baseline are *small* (hundredths of RMSE) on this small dataset. Netflix-Prize-scale gains came from huge data, tuning and ensembles.
- Therefore: **always report a baseline**, and don't trust a 0.002 difference without repeating over seeds (see §8).

## 6. Open the model: what did SVD learn?
$P$ = `svd.pu` (users × k), $Q$ = `svd.qi` (items × k), biases `svd.bu`, `svd.bi`.

In [ ]:
print("P (users×factors):", svd.pu.shape, "| Q (items×factors):", svd.qi.shape, "| μ =", round(svd.trainset.global_mean, 3))
print("Parameters learned:", svd.pu.size + svd.qi.size + len(svd.bu) + len(svd.bi), "vs the full matrix:", trainset.n_users * trainset.n_items)

# Re-compute one prediction by hand:  μ + b_u + b_i + p_u · q_i
uid, iid = 10, 1                                   # user 10, Toy Story
iu, ii = trainset.to_inner_uid(uid), trainset.to_inner_iid(iid)
manual = svd.trainset.global_mean + svd.bu[iu] + svd.bi[ii] + svd.pu[iu] @ svd.qi[ii]
print(f"\nuser {uid} × '{title[iid]}':  by hand = {manual:.4f}   svd.predict = {svd.predict(uid, iid).est:.4f}")
print(f"   μ={svd.trainset.global_mean:.2f}  b_u={svd.bu[iu]:+.2f}  b_i={svd.bi[ii]:+.2f}  p_u·q_i={svd.pu[iu] @ svd.qi[ii]:+.2f}")

In [ ]:
# Biases: the most "generous" movies and harshest users
raw_i = np.array([trainset.to_raw_iid(k) for k in range(trainset.n_items)]); cnt = np.bincount([i for (_, i, _) in trainset.all_ratings()], minlength=trainset.n_items)
bias_df = pd.DataFrame({"movieId": raw_i, "b_i": svd.bi, "n": cnt}); bias_df["title"] = title.reindex(bias_df.movieId).values
print("Movies with the highest item bias (≥ 50 ratings):"); display(bias_df[bias_df.n >= 50].nlargest(5, "b_i")[["title", "b_i", "n"]].round(2))
print("…and the lowest:"); bias_df[bias_df.n >= 50].nsmallest(5, "b_i")[["title", "b_i", "n"]].round(2)

### Latent factors → "similar movies" without any genre information
Items are now **vectors**; similar movies = nearby vectors (cosine on rows of $Q$). Compare with Lab 2's neighbours.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
Q = svd.qi; counts = pd.Series(cnt, index=raw_i)
def similar_by_factors(movie_id, n=8, min_n=30):
    s = cosine_similarity(Q[[trainset.to_inner_iid(movie_id)]], Q)[0]
    out = pd.DataFrame({"movieId": raw_i, "cos": s, "n": cnt}); out = out[(out.n >= min_n) & (out.movieId != movie_id)]
    out["title"] = title.reindex(out.movieId).values
    return out.nlargest(n, "cos")[["title", "cos", "n"]].round(3)
for mid in (1, 2571):
    print("Closest to", title[mid]); display(similar_by_factors(mid))

In [ ]:
# 2-D picture of the item space (PCA of Q for the 300 most-rated movies), coloured by main genre
from sklearn.decomposition import PCA
top = np.argsort(-cnt)[:300]; xy = PCA(2, random_state=0).fit_transform(Q[top])
main = movies.reindex(raw_i[top]).genres.str.split("|").str[0].values
fig, ax = plt.subplots(figsize=(7, 5)); genres = ["Drama", "Comedy", "Action", "Adventure", "Thriller", "Crime", "Sci-Fi", "Animation"]
for gname in genres:
    m_ = main == gname; ax.scatter(xy[m_, 0], xy[m_, 1], s=22, label=gname, alpha=.8)
ax.scatter(xy[~np.isin(main, genres), 0], xy[~np.isin(main, genres), 1], s=10, c="lightgray", label="other")
for mid in (1, 2571, 318, 296, 4993, 1721):
    j = np.where(raw_i[top] == mid)[0]
    if len(j): ax.annotate(title[mid].split(" (")[0], xy[j[0]], fontsize=8)
ax.legend(fontsize=7, ncol=2); ax.set(title="Item factors Q projected to 2-D (300 most-rated movies)", xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.savefig(FIG / "lab3_item_space.png"); plt.show()

**Compare with Lab 2:** adjusted-cosine neighbours were cleaner than these. The latent space is a *compression* trained to predict ratings, not to be a similarity engine — expect some surprises (and some real structure).

⚠️ **Caution (say this out loud):** latent dimensions are *not* guaranteed to equal genres. Any rotation of the factors gives the same predictions, so individual axes have no fixed meaning — only the *geometry* (who is near whom) is meaningful.

## 7. Top-N recommendations
Prediction → **ranking**: predict a rating for every movie the user has *not* rated, sort, take the top N.
```
User → candidate movies (unrated) → predict rating for each → sort → Top-N
```

In [ ]:
all_iids = list(trainset._raw2inner_id_items.keys())     # every movie in the training data
counts_by_raw = counts

def top_n(algo, uid, n=5, min_pop=20):
    seen = {trainset.to_raw_iid(i) for i, _ in trainset.ur[trainset.to_inner_uid(uid)]}
    cands = [i for i in all_iids if i not in seen and counts_by_raw[i] >= min_pop]   # the same support guard as Lab 2
    scored = sorted(((i, algo.predict(uid, i).est) for i in cands), key=lambda t: -t[1])[:n]
    return pd.DataFrame({"title": [title[i] for i, _ in scored], "predicted": [round(p, 2) for _, p in scored],
                         "# ratings": [counts_by_raw[i] for i, _ in scored]}, index=[i for i, _ in scored])

print("Recommended for user 10 (SVD, 50 factors):"); top_n(svd, 10)

In [ ]:
print("User 10's own favourites in training, for comparison:")
u10 = train[train.userId == 10].nlargest(8, "rating"); u10.assign(title=title.reindex(u10.movieId).values)[["title", "rating"]]

In [ ]:
# Three different users → three different lists (personalisation!)
for u in (10, 414, 77):
    print(f"\n=== user {u} ({(train.userId == u).sum()} training ratings) ==="); display(top_n(svd, u, n=5))

## 8. Experiment: how many latent factors?  (research-style)
> *Does increasing `n_factors` always improve performance?*

One run per setting is **noise** (random initialisation) → repeat over 5 seeds and report **mean ± std**.

In [ ]:
rows = []
for k in (5, 20, 50, 100, 200):
    for seed in range(5):
        m = SVD(n_factors=k, random_state=seed).fit(trainset); p = m.test(testset)
        rows.append({"n_factors": k, "seed": seed, "RMSE": accuracy.rmse(p, verbose=False), "MAE": accuracy.mae(p, verbose=False)})
exp = pd.DataFrame(rows); summ = exp.groupby("n_factors").agg(RMSE=("RMSE", "mean"), RMSE_std=("RMSE", "std"), MAE=("MAE", "mean"))
display(summ.round(4))
fig, ax = plt.subplots(figsize=(5.8, 3.4)); ax.errorbar(summ.index, summ.RMSE, yerr=summ.RMSE_std, fmt="o-", capsize=3, color="#2b6cb0")
ax.axhline(results["baseline  μ + b_u + b_i"]["RMSE"], ls="--", color="gray", label="bias baseline"); ax.legend()
ax.set(xlabel="n_factors", ylabel="test RMSE", title="Effect of latent dimension (mean ± std over 5 seeds)", xscale="log")
plt.tight_layout(); plt.savefig(FIG / "lab3_nfactors.png"); plt.show()

**Discuss:** the curve is *flat or slowly rising* — more factors ≠ better. More capacity means more room to **overfit** sparse ratings (about 130 ratings per user, 8 per movie). The remedy is *regularisation* (`reg_all`), not just fewer factors. Let's tune it.

In [ ]:
# Small grid search with 3-fold CV on the TRAINING set only (the test set stays untouched!)
grid = {"n_factors": [20, 50], "n_epochs": [20, 40], "lr_all": [0.005, 0.01], "reg_all": [0.02, 0.05, 0.1]}
gs = GridSearchCV(SVD, grid, measures=["rmse"], cv=3, n_jobs=-1, joblib_verbose=0); gs.fit(data)
print("best CV RMSE:", round(gs.best_score["rmse"], 4), "with", gs.best_params["rmse"])
cv = pd.DataFrame(gs.cv_results)[["param_n_factors", "param_n_epochs", "param_lr_all", "param_reg_all", "mean_test_rmse"]].sort_values("mean_test_rmse")
cv.head(6).round(4)

In [ ]:
best = SVD(**gs.best_params["rmse"], random_state=42)
best, _ = run("SVD tuned (CV on train)", best)
pd.DataFrame(results).T.round(4)

Tuning `reg_all` helps more than adding factors. **Never tune on the test set**: we picked the parameters with cross-validation on *train*, then looked at test once.

## 9. Rating prediction ≠ Top-N recommendation
A model with a great RMSE can still produce a poor top-10 list. Evaluate the *ranking* directly:
for every test user, rank all their unseen movies, take the top 10, and ask how many are movies the user later **liked (rating ≥ 4)** in the test set.

- **Precision@10** = (# liked items in top-10) / 10 · **Recall@10** = (# liked in top-10) / (# liked in test)
- We compare SVD with a **popularity** recommender (same list for everyone). *Candidate set = all movies with ≥ 20 training ratings.*

In [ ]:
THRESH, K = 4.0, 10
liked = test[test.rating >= THRESH].groupby("userId").movieId.apply(set)
train_seen = train.groupby("userId").movieId.apply(set)
cands = [i for i in all_iids if counts_by_raw[i] >= 20]
pop = train.groupby("movieId").rating.agg(["mean", "count"]); pop = pop.loc[cands]
pop["score"] = (pop["count"] * pop["mean"] + 30 * train.rating.mean()) / (pop["count"] + 30)
pop_rank = list(pop.sort_values("score", ascending=False).index)

def pr_at_k(rank_fn, users, K=K):
    P, R_ = [], []
    for u in users:
        rec = rank_fn(u)[:K]; hit = len(set(rec) & liked[u])
        P.append(hit / K); R_.append(hit / len(liked[u]))
    return np.mean(P), np.mean(R_)

eval_users = [u for u in liked.index if len(liked[u]) >= 3][:300]          # 300 users keeps it quick
def rank_pop(u): return [i for i in pop_rank if i not in train_seen[u]]
def rank_model(algo):
    def f(u):
        s = [(i, algo.predict(u, i).est) for i in cands if i not in train_seen[u]]
        return [i for i, _ in sorted(s, key=lambda t: -t[1])]
    return f

t = time.time()
rank_res = pd.DataFrame({"popularity (non-personalised)": pr_at_k(rank_pop, eval_users),
                         "SVD (50 factors)": pr_at_k(rank_model(svd), eval_users),
                         "SVD tuned": pr_at_k(rank_model(best), eval_users)}, index=[f"Precision@{K}", f"Recall@{K}"]).T
print(f"evaluated {len(eval_users)} users in {time.time() - t:.0f}s"); rank_res.round(4)

**Why does SVD lose here?** (1) It is trained to minimise *rating* error, not to *rank*; (2) candidates with few ratings can get inflated predictions (noisy item factors); (3) "not in the test set" ≠ "disliked" — unobserved items are *missing not at random*, which favours popular items. Try raising the candidate threshold to ≥ 100 ratings and watch Precision@10 change.

**Interpretation.** Popularity is a *brutally* strong top-N baseline on MovieLens (popular movies are watched *and* liked by most). A personalised model that wins on RMSE does not automatically win on Precision@10 — **they measure different things**:

| | rating prediction | Top-N recommendation |
|---|---|---|
| question | "what stars will she give?" | "which 10 should we show?" |
| metric | RMSE / MAE | Precision@N, Recall@N, NDCG |
| depends on | accuracy of *every* prediction | *ordering* at the top |

Real systems also care about diversity, novelty and fairness — hints for Day 2.

## 10. (Optional) ALS from scratch
**ALS**: fix $Q$ → each user vector $p_u$ is a *ridge regression* with a closed form; fix $P$ → same for each item; alternate.
$$p_u=\Big(\sum_{i\in\Omega_u}q_iq_i^\top+\lambda I\Big)^{-1}\sum_{i\in\Omega_u}r_{ui}q_i$$
No learning rate needed; each half-step solves a small $k\times k$ system exactly.

In [ ]:
def als(train_df, k=20, lam=10.0, iters=12, seed=0):
    uids = np.sort(train_df.userId.unique()); iids = np.sort(train_df.movieId.unique())
    ui = {u: n for n, u in enumerate(uids)}; ii = {i: n for n, i in enumerate(iids)}
    u_ = train_df.userId.map(ui).values; i_ = train_df.movieId.map(ii).values; r_ = train_df.rating.values
    mu = r_.mean(); r0 = r_ - mu                                   # centre globally; ALS learns the rest
    by_u = [np.where(u_ == n)[0] for n in range(len(uids))]; by_i = [np.where(i_ == n)[0] for n in range(len(iids))]
    rng = np.random.default_rng(seed); P = rng.normal(0, .1, (len(uids), k)); Q = rng.normal(0, .1, (len(iids), k)); I = np.eye(k)
    hist = []
    for it in range(iters):
        for n, idx in enumerate(by_u):                              # fix Q, solve each user
            Qi = Q[i_[idx]]; P[n] = np.linalg.solve(Qi.T @ Qi + lam * I, Qi.T @ r0[idx])
        for n, idx in enumerate(by_i):                              # fix P, solve each item
            Pu = P[u_[idx]]; Q[n] = np.linalg.solve(Pu.T @ Pu + lam * I, Pu.T @ r0[idx])
        hist.append(np.sqrt(np.mean((r0 - (P[u_] * Q[i_]).sum(1)) ** 2)))
    return dict(P=P, Q=Q, mu=mu, ui=ui, ii=ii, hist=hist)

def als_predict(m, df):
    out = np.full(len(df), m["mu"])
    ok = df.userId.isin(m["ui"]).values & df.movieId.isin(m["ii"]).values
    uu = df.userId[ok].map(m["ui"]).values; vv = df.movieId[ok].map(m["ii"]).values
    out[ok] = m["mu"] + (m["P"][uu] * m["Q"][vv]).sum(1)
    return out.clip(0.5, 5)

t = time.time(); m_als = als(train, k=20, lam=10.0, iters=12)
p = als_predict(m_als, test)
print(f"ALS (k=20, λ=10): test RMSE = {np.sqrt(np.mean((test.rating - p) ** 2)):.4f}  MAE = {np.mean(np.abs(test.rating - p)):.4f}   [{time.time() - t:.0f}s]")
print("train RMSE per sweep:", np.round(m_als['hist'], 3))

Training error falls monotonically with every sweep (each half-step is an exact minimiser — **the objective can never go up**). With no bias terms this plain ALS is less accurate than biased SVD: add $b_u,b_i$ (or use `BaselineOnly` first) as an exercise. Try `lam` ∈ {1, 10, 30}: too small overfits, too large underfits.

## 11. 🏁 Final challenge — *independently*
> **For a given user, build a recommender that recommends five movies the user has not rated.**

Write `recommend(user_id, n=5)` using any model from today, return titles, and justify your choice of model in one sentence.

In [ ]:
# TODO: write your code here


## 12. Summary table & Day-1 pipeline

In [ ]:
final = pd.DataFrame(results).T[["RMSE", "MAE"]].round(4); final.sort_values("RMSE")

```
Raw ratings ─▶ EDA ─▶ user–item matrix ─▶ similarity ─▶ neighbourhood CF ─▶ matrix factorization (SVD / ALS)
                                                                         ─▶ rating prediction ─▶ Top-N ─▶ RMSE/MAE, Precision@N
```
## ✅ Lab 3 deliverable — checklist
- [x] Surprise dataset & split · [x] SVD trained · [x] predictions table · [x] RMSE & MAE vs baselines
- [x] inspected $P$, $Q$, biases · [x] Top-5 for user 10 · [x] `n_factors` experiment (5 seeds) · [x] tuning with CV
- [x] Precision@10 vs popularity · [x] (optional) ALS

**Next:** Day 2 builds on this pipeline.